# Extended Hybrid Random Forest Tuning (Analysis F)

## Protocol

**Status:** written on 2026-10-22, before any analysis code in this notebook was run. Under the project's commit policy it is not committed separately before execution; it is committed with the completed work package. Before execution the code was checked only by a smoke test with **randomly permuted grades** on two outer folds, which reveals no result.

### 1. Framing

In notebook 13, the hybrid Random Forest's selected settings reached the edge of the search space for two settings:

- `max_features = 0.3`, the larger of the two values tested, was chosen in **all 25** outer folds;
- 32 PCA components, the largest value tested, was chosen in 16 of 25 folds.

So better settings may lie beyond the grid. This analysis asks:

> **Does extending the PCA size and `max_features` improve the tuned hybrid Random Forest?**

It is a **follow-up**, not an independent confirmation:

- The outer folds are the same donor-grouped folds used in notebooks 09–14. They have informed earlier decisions, including this question, which came from inspecting notebook 13's selections.
- This is the third tuning-related analysis on these folds, after notebooks 13 and 14. Each further search on the same folds makes a favourable result more likely by chance.
- Independent confirmation requires new, independently graded tissues.

Existing results are not changed. The only reproduction is the reference check in Section 4.

### 2. Cohort

As notebooks 13 and 14, unchanged.

| Set | Use | Expected |
|---|---|---|
| Primary cohort K3: non-2023, image-matched, year from `Tissue_ID` | Training and inner tuning | 551 tissues, 312 donors |
| ECD evaluation set K4: K3 with ECD | Outer-fold evaluation; training of the ECD tree | 470 tissues, 280 donors; A+ 244, A 159, B 67 |

The notebook asserts these counts and stops if they differ.

### 3. Validation design

As notebooks 13 and 14, unchanged:

- **Outer folds:** `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to K3 (25 outer folds).
- **Inner folds:** `StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)` within each outer training fold, grouped by inferred donor. Every inner validation fold must contain all three grades.
- **Inner selection metric:** mean macro F1 over the 3 inner validation folds. Ties go to the first setting in grid order.
- **Leakage controls:**
  - all preprocessing, including the PCA, is fitted on inner-training data for selection and on the full outer-training fold for the refit;
  - the PCA never sees validation or test tissues;
  - the outer test fold is used only for the final prediction.

The candidate is tuned in parallel worker processes, one per outer fold, each limited to one thread. The reference models are fitted in the main process with default threading, as in notebook 13 (see notebook 14's deviations log: PCA rounding depends on the thread count).

### 4. Models

**References (reproduced, not re-tuned):**

| Model | How it is obtained | Reproduction check |
|---|---|---|
| ECD tree | Depth-2 tree on ECD, as notebooks 11, 13 and 14 | Fold scores equal `fitted_ecd_fold_scores.csv` |
| **Tuned RF hybrid** (deployed family, notebook 13) | Refit, in each outer fold, of the setting notebook 13 selected for that fold | Fold scores equal `nested_tuning_fold_scores.csv` |

Each check requires a maximum absolute difference below 1e-9 in every metric, or the notebook stops.

**Candidate: extended RF hybrid.**

- **Inputs and preprocessing:** the same as notebook 13's hybrid, so that **only the search space differs**:
  - the 24 tabular features through `ms.make_preprocessor()`;
  - the 2048 ResNet-50 embedding dimensions, standardised, then PCA (`random_state=42`), with no further scaling.
- **Fixed forest settings:** `RandomForestClassifier(n_estimators=300, max_depth=15, criterion="gini", random_state=42)`.

Search space (grid order = tie-break order):

| Setting | Values |
|---|---|
| PCA components | 16, 32, 64, 128 |
| `max_features` | 0.3, 0.5, 0.7 |
| `min_samples_leaf` | 2, 1 |
| `class_weight` | balanced, None |

That is 48 settings. 0.3 and PCA 16/32 overlap notebook 13's grid, so the extended search can fall back on them.

`max_depth` is fixed at 15, the value chosen in 20 of 25 folds in notebook 13. PCA 8 is dropped: it was chosen in only 2 folds.

### 5. Outcomes and comparisons

**Metrics** for every model on the K4 tissues of each outer test fold:

- macro F1 (**primary**);
- balanced accuracy;
- accuracy;
- grade B recall;
- quadratic-weighted Cohen's kappa, with the order B < A < A+.

Fold metrics are reported as mean ± SD over 25 folds. Per-grade precision and recall and kappa are also reported pooled over repeats.

**Primary comparison (one test):** extended RF hybrid − notebook 13 tuned RF hybrid, macro F1.

**Secondary comparison:** extended RF hybrid − ECD tree, macro F1.

**Secondary metrics:** differences in balanced accuracy, accuracy, grade B recall and kappa for both comparisons, unadjusted.

**Test:** the corrected repeated-CV t-test, as in notebooks 09–14. It uses the paired differences over 25 outer folds, 24 degrees of freedom, and the K3 training and test fold sizes. The 95% CIs are unadjusted.

**Verdict for each comparison:**

| Verdict | Condition |
|---|---|
| Improved | difference > 0 and p < 0.05 |
| Worse | difference < 0 and p < 0.05 |
| Inconclusive | otherwise |

There is one primary comparison, so no multiplicity correction is applied to it. Repeated analyses on these folds (notebooks 13–15) are a limitation that p-values do not correct for (Section 1).

**Descriptive only:**

- the distribution of selected settings, in particular **whether selections now reach the new edges** (PCA 128, `max_features` 0.7);
- the inner macro F1.

### 6. Application model rule

Fixed here and applied mechanically.

The extended hybrid **convincingly outperforms** the deployed model if:

- its macro F1 difference from the notebook 13 tuned RF hybrid is > 0;
- p < 0.05;
- its grade B recall difference is ≥ −0.02.

- If it does not, **the deployed tuned RF hybrid is retained.**
- If it does, a replacement is **recommended**. Retraining, exporting and deploying would be a separate, documented task, and the result would be stated as a hypothesis requiring confirmation on new tissues.

This notebook does not retrain, export or deploy any model. A higher point estimate without a convincing improvement does not change the deployed model.

### 7. Interpretation rules

- **A higher point estimate is not an improvement.** Only the verdicts in Section 5 are reported as improvements or deteriorations.
- **An inconclusive result means the data cannot distinguish the two tuning spaces.** It is not evidence that they are equivalent.
- **Edge selections:** if selections cluster at PCA 128 or `max_features` 0.7, that is reported as a reason for further search, not as evidence of improvement.
- **No re-runs to change results.** The notebook is executed once. Re-runs are allowed only to fix errors that do not change any choice in this protocol, and each is recorded in the deviations log.

# Analysis

Implements the protocol above without changes. Section numbers refer to the protocol.

In [1]:
import hashlib
import json
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=UserWarning)

SCRIPTS_DIR = str(Path("../../scripts").resolve())
sys.path.insert(0, SCRIPTS_DIR)
os.environ["PYTHONPATH"] = SCRIPTS_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")  # for worker processes
import modeling_setup as ms

SEED = ms.RANDOM_STATE
CLASSES = list(ms.CLASSES)
ORDINAL = ["B", "A", "A+"]  # kappa weights follow this order
ECD = "ECD_CD_cells_per_mm2"
TABULAR_FEATURES = list(ms.ALL_FEATURES)
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB11_SCORES_PATH = ms.PROCESSED_DIR / "fitted_ecd_fold_scores.csv"
NB13_SCORES_PATH = ms.PROCESSED_DIR / "nested_tuning_fold_scores.csv"
NB13_SELECTIONS_PATH = ms.PROCESSED_DIR / "nested_tuning_selections.csv"
REVIEW_IDS = ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]
INNER_SPLITS = 3
N_WORKERS = min(12, os.cpu_count())
TREE_PARAMS = dict(max_depth=2, min_samples_leaf=10, class_weight="balanced", criterion="gini", random_state=SEED)

# Protocol Section 4. Grid order is the tie-break order.
PCA_GRID = [16, 32, 64, 128]
FOREST_GRID = [
    {"max_features": features, "min_samples_leaf": leaf, "class_weight": weight}
    for features in [0.3, 0.5, 0.7]
    for leaf in [2, 1]
    for weight in ["balanced", None]
]
FOREST_FIXED = dict(n_estimators=300, max_depth=15, criterion="gini", random_state=SEED)
assert len(PCA_GRID) * len(FOREST_GRID) == 48

CANDIDATE = "Extended RF hybrid"
MODELS = ["ECD tree", "Tuned RF hybrid", CANDIDATE]

for path in [ms.DATA_PATH, ms.DONOR_FOLDS_PATH, EMBEDDING_PATH, NB11_SCORES_PATH, NB13_SCORES_PATH, NB13_SELECTIONS_PATH]:
    print(f"{path.name}: sha256 {hashlib.sha256(path.read_bytes()).hexdigest()[:16]}…")

cleaned_grading_dataset.xlsx: sha256 0ce9b384a51a4c47…
donor_grouped_folds.csv: sha256 755280ff9d7bd388…
resnet50_imagenet_fullres.npz: sha256 883d4a1723a17be7…
fitted_ecd_fold_scores.csv: sha256 fe0c10e5d3431501…
nested_tuning_fold_scores.csv: sha256 ad9536db279af708…
nested_tuning_selections.csv: sha256 7ff3077bc130fa81…


## Cohort (Protocol Section 2)

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin({normalize_id(i) for i in REVIEW_IDS})]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
labelled["Year"] = labelled["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype("Int64")

with_image = labelled.merge(manifest[["Match_ID"]], on="Match_ID", how="inner", validate="one_to_one")
cohort = (
    with_image[with_image["Year"].notna() & (with_image["Year"] != 2023)]
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)
has_ecd = cohort[ECD].notna().to_numpy()
evaluation = cohort[has_ecd]

primary_desc = {"tissues": len(cohort), "donors": cohort["Donor_Group"].nunique()}
eval_desc = {
    "tissues": len(evaluation), "donors": evaluation["Donor_Group"].nunique(),
    "grades": {k: int(v) for k, v in evaluation[ms.TARGET].value_counts().items()},
}
print("K3:", primary_desc, "| K4:", eval_desc)
assert primary_desc == {"tissues": 551, "donors": 312}, "K3 differs from protocol"
assert eval_desc == {"tissues": 470, "donors": 280, "grades": {"A+": 244, "A": 159, "B": 67}}, "K4 differs from protocol"

cached = np.load(EMBEDDING_PATH, allow_pickle=True)
row_of = {tissue_id: i for i, tissue_id in enumerate(cached["tissue_ids"])}
embeddings = cached["embeddings"][[row_of[t] for t in cohort["Tissue_ID"]]]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[TABULAR_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()
donors = cohort["Donor_Group"].to_numpy()
print("Feature matrix:", X.shape)

K3: {'tissues': 551, 'donors': 312} | K4: {'tissues': 470, 'donors': 280, 'grades': {'A+': 244, 'A': 159, 'B': 67}}
Feature matrix: (551, 2072)


## Models and tuning procedure (Protocol Sections 3–4)

In [3]:
TABULAR_PREPROCESSOR = ms.make_preprocessor()


def rf_preprocessor(n_components):
    # Notebook 13's hybrid preprocessing, unchanged; used by the reference and the candidate.
    branches = [("tabular", clone(TABULAR_PREPROCESSOR), TABULAR_FEATURES)]
    if n_components:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_components, random_state=SEED)),
        ]), EMBEDDING_COLUMNS))
    return ColumnTransformer(branches, remainder="drop")


def macro_f1(true, pred):
    return f1_score(true, pred, labels=CLASSES, average="macro", zero_division=0)


def tune(rows, X, y, donors):
    # Select a setting by 3-fold donor-grouped inner CV on `rows` (positions in the cohort).
    candidates = [(k, s) for k in PCA_GRID for s in FOREST_GRID]
    scores = np.zeros((INNER_SPLITS, len(candidates)))
    splitter = StratifiedGroupKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    for i, (inner_train, inner_val) in enumerate(splitter.split(rows, y[rows], groups=donors[rows])):
        tr, va = rows[inner_train], rows[inner_val]
        assert set(y[va]) == set(CLASSES), "An inner validation fold lacks a grade"
        assert set(donors[tr]).isdisjoint(donors[va]), "Inner donor leak"
        for k in PCA_GRID:
            pre = rf_preprocessor(k).fit(X.iloc[tr], y[tr])
            Z_tr, Z_va = pre.transform(X.iloc[tr]), pre.transform(X.iloc[va])
            for j, (kk, setting) in enumerate(candidates):
                if kk != k:
                    continue
                forest = RandomForestClassifier(n_jobs=1, **FOREST_FIXED, **setting)
                scores[i, j] = macro_f1(y[va], forest.fit(Z_tr, y[tr]).predict(Z_va))
    mean_scores = scores.mean(axis=0)
    best = int(np.argmax(mean_scores))  # first maximum = tie-break order
    return candidates[best], mean_scores[best]


def run_outer_fold(repeat, fold, train, test, X, y, donors, has_ecd):
    # Candidate tuning for one outer fold, in a worker process limited to one thread.
    with threadpool_limits(1):
        assert set(donors[train]).isdisjoint(donors[test]), "Outer donor leak"
        eval_test = test[has_ecd[test]]
        assert set(y[eval_test]) == set(CLASSES), "An outer test fold lacks a grade"
        (k, setting), inner_score = tune(train, X, y, donors)
        model = Pipeline([
            ("features", rf_preprocessor(k)),
            ("classifier", RandomForestClassifier(n_jobs=1, **FOREST_FIXED, **setting)),
        ])
        prediction = model.fit(X.iloc[train], y[train]).predict(X.iloc[eval_test])
    return {"repeat": repeat, "fold": fold, "eval_test": eval_test, "predictions": {CANDIDATE: prediction},
            "selection": {"Repeat": repeat, "Fold": fold, "PCA": k, **setting, "Inner macro F1": inner_score},
            "sizes": (len(train), len(test))}


# Notebook 13's selected hybrid setting for each outer fold (Protocol Section 4).
nb13_selections = pd.read_csv(NB13_SELECTIONS_PATH, dtype={"max_features": str})
nb13_hybrid = nb13_selections[nb13_selections["Arm"] == "Tuned hybrid"]
rf_settings = {
    (int(row["Repeat"]), int(row["Fold"])): (
        int(row["PCA"]),
        {
            "min_samples_leaf": int(row["min_samples_leaf"]),
            "max_depth": None if pd.isna(row["max_depth"]) else int(row["max_depth"]),
            "max_features": row["max_features"] if row["max_features"] == "sqrt" else float(row["max_features"]),
            "class_weight": None if pd.isna(row["class_weight"]) else row["class_weight"],
        },
    )
    for row in nb13_hybrid.to_dict("records")
}
assert len(rf_settings) == 25, "Expected 25 outer folds"


def run_references(repeat, fold, train, test):
    # Reference models, fitted in the main process with default BLAS threading as in notebook 13.
    eval_test = test[has_ecd[test]]
    tree = Pipeline([
        ("features", ColumnTransformer([("ecd", "passthrough", [ECD])], remainder="drop")),
        ("classifier", DecisionTreeClassifier(**TREE_PARAMS)),
    ]).fit(X.iloc[train[has_ecd[train]]], y[train[has_ecd[train]]])
    k, setting = rf_settings[(repeat, fold)]
    forest = Pipeline([
        ("features", rf_preprocessor(k)),
        ("classifier", RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, **setting)),
    ]).fit(X.iloc[train], y[train])
    return {"ECD tree": tree.predict(X.iloc[eval_test]), "Tuned RF hybrid": forest.predict(X.iloc[eval_test])}

## Outer cross-validation (25 folds, run in parallel)

In [4]:
METRICS = {
    "Macro F1": macro_f1,
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
    "Recall B": lambda t, p: recall_score(t, p, labels=["B"], average="macro", zero_division=0),
    "Kappa (quadratic)": lambda t, p: cohen_kappa_score(t, p, labels=ORDINAL, weights="quadratic"),
}

jobs = [
    (repeat, fold, train, test)
    for repeat in range(ms.N_REPEATS)
    for fold, (train, test) in enumerate(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
]
assert len(jobs) == 25

start = time.time()
outputs = Parallel(n_jobs=N_WORKERS, backend="loky")(
    delayed(run_outer_fold)(repeat, fold, train, test, X, y, donors, has_ecd)
    for repeat, fold, train, test in jobs
)
for out, (repeat, fold, train, test) in zip(outputs, jobs):
    assert (out["repeat"], out["fold"]) == (repeat, fold)
    out["predictions"] = {**run_references(repeat, fold, train, test), **out["predictions"]}
print(f"Total time {(time.time() - start) / 60:.1f} min on {N_WORKERS} processes")

fold_rows, fold_sizes = [], []
oof = {(m, r): np.full(len(cohort), None, dtype=object) for m in MODELS for r in range(ms.N_REPEATS)}
for out in outputs:
    fold_sizes.append(out["sizes"])
    for model, pred in out["predictions"].items():
        oof[(model, out["repeat"])][out["eval_test"]] = pred
        fold_rows.append({"Model": model, "Repeat": out["repeat"], "Fold": out["fold"], "N": len(out["eval_test"]),
                          **{m: fn(y[out["eval_test"]], pred) for m, fn in METRICS.items()}})

fold_scores = pd.DataFrame(fold_rows)
selections = pd.DataFrame([out["selection"] for out in outputs])
N_TRAIN, N_TEST = np.mean(fold_sizes, axis=0)

Total time 12.3 min on 12 processes


### Reproduction checks (Protocol Section 4)

In [5]:
CHECKED = ["Macro F1", "Balanced accuracy", "Accuracy", "Recall B"]
for ours, path, theirs in [
    ("ECD tree", NB11_SCORES_PATH, "ECD tree"),
    ("Tuned RF hybrid", NB13_SCORES_PATH, "Tuned hybrid"),
]:
    saved = pd.read_csv(path)
    a = fold_scores[fold_scores["Model"] == ours].set_index(["Repeat", "Fold"])[CHECKED].sort_index()
    b = saved[saved["Model"] == theirs].set_index(["Repeat", "Fold"])[CHECKED].sort_index()
    assert len(a) == len(b) == 25, f"{ours}: expected 25 folds"
    max_diff = (a - b).abs().to_numpy().max()
    print(f"{ours}: max absolute difference from {path.name} = {max_diff:.2e}")
    assert max_diff < 1e-9, f"{ours} not reproduced: stop and record in the deviations log"

ECD tree: max absolute difference from fitted_ecd_fold_scores.csv = 0.00e+00
Tuned RF hybrid: max absolute difference from nested_tuning_fold_scores.csv = 5.55e-17


## Results (Protocol Section 5)

In [6]:
summary = fold_scores.groupby("Model")[list(METRICS)].agg(["mean", "std"]).reindex(MODELS)
print("Outer-fold metrics on K4 tissues, mean and SD over 25 folds:")
print(summary.round(3).to_string())

Outer-fold metrics on K4 tissues, mean and SD over 25 folds:
                   Macro F1        Balanced accuracy        Accuracy        Recall B        Kappa (quadratic)       
                       mean    std              mean    std     mean    std     mean    std              mean    std
Model                                                                                                               
ECD tree              0.796  0.038             0.768  0.043    0.804  0.031    0.790  0.117             0.771  0.056
Tuned RF hybrid       0.793  0.047             0.775  0.050    0.797  0.040    0.789  0.130             0.764  0.066
Extended RF hybrid    0.802  0.035             0.781  0.041    0.803  0.029    0.793  0.115             0.770  0.052


In [7]:
def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, t, p


wide = fold_scores.pivot_table(index=["Repeat", "Fold"], columns="Model", values=list(METRICS))
COMPARISONS = {"F1": ("Primary", "Tuned RF hybrid"), "F2": ("Secondary", "ECD tree")}
rows = []
for label, (role, baseline) in COMPARISONS.items():
    for metric in METRICS:
        d = wide[(metric, CANDIDATE)] - wide[(metric, baseline)]
        mean, low, high, t, p = corrected_ttest(d)
        verdict = ""
        if metric == "Macro F1":
            verdict = "Improved" if (p < 0.05 and mean > 0) else "Worse" if (p < 0.05 and mean < 0) else "Inconclusive"
        rows.append({
            "ID": label, "Role": role, "Comparison": f"{CANDIDATE} − {baseline}", "Metric": metric,
            "Model": wide[(metric, CANDIDATE)].mean(), "Baseline": wide[(metric, baseline)].mean(),
            "Difference": mean, "CI low": low, "CI high": high, "t": t, "p": p,
            "Folds improved": f"{(d > 0).sum()}/{len(d)}", "Verdict": verdict,
        })
results = pd.DataFrame(rows)

print("Macro F1 comparisons (CIs unadjusted):")
print(results[results["Metric"] == "Macro F1"].drop(columns="Metric").round(4).to_string(index=False))
print("\nSecondary metrics (unadjusted):")
print(results[results["Metric"] != "Macro F1"].drop(columns="Verdict").round(4).to_string(index=False))

Macro F1 comparisons (CIs unadjusted):
ID      Role                           Comparison  Model  Baseline  Difference  CI low  CI high      t      p Folds improved      Verdict
F1   Primary Extended RF hybrid − Tuned RF hybrid 0.8018    0.7934      0.0085 -0.0160   0.0330 0.7159 0.4810          13/25 Inconclusive
F2 Secondary        Extended RF hybrid − ECD tree 0.8018    0.7958      0.0060 -0.0187   0.0307 0.5032 0.6194          14/25 Inconclusive

Secondary metrics (unadjusted):
ID      Role                           Comparison            Metric  Model  Baseline  Difference  CI low  CI high       t      p Folds improved
F1   Primary Extended RF hybrid − Tuned RF hybrid Balanced accuracy 0.7805    0.7750      0.0055 -0.0142   0.0252  0.5741 0.5712          12/25
F1   Primary Extended RF hybrid − Tuned RF hybrid          Accuracy 0.8029    0.7971      0.0058 -0.0148   0.0263  0.5788 0.5681          13/25
F1   Primary Extended RF hybrid − Tuned RF hybrid          Recall B 0.7931    0.78

### Descriptive results

In [8]:
eval_rows = np.flatnonzero(has_ecd)
y_eval = np.tile(y[eval_rows], ms.N_REPEATS)
pooled = {m: np.concatenate([oof[(m, r)][eval_rows] for r in range(ms.N_REPEATS)]).astype(str) for m in MODELS}
assert all(len(p) == len(y_eval) and "None" not in set(p) for p in pooled.values())

per_grade = pd.concat({
    m: pd.DataFrame({
        "Precision": precision_score(y_eval, pooled[m], labels=CLASSES, average=None, zero_division=0),
        "Recall": recall_score(y_eval, pooled[m], labels=CLASSES, average=None, zero_division=0),
    }, index=CLASSES).stack()
    for m in MODELS
}, axis=1).T
per_grade["Kappa (quadratic)"] = [cohen_kappa_score(y_eval, pooled[m], labels=ORDINAL, weights="quadratic") for m in MODELS]
print("Per-grade precision and recall, and quadratic-weighted kappa, on K4 pooled over repeats:")
print(per_grade.round(3).to_string())

Per-grade precision and recall, and quadratic-weighted kappa, on K4 pooled over repeats:
                          A+                A                B        Kappa (quadratic)
                   Precision Recall Precision Recall Precision Recall                  
ECD tree               0.748  0.988     0.884  0.528     1.000  0.791             0.772
Tuned RF hybrid        0.788  0.906     0.761  0.635     0.936  0.791             0.769
Extended RF hybrid     0.786  0.913     0.773  0.639     0.964  0.794             0.774


In [9]:
print("Selected settings across 25 outer folds (Protocol Section 5, descriptive):")
display_selections = selections.assign(class_weight=selections["class_weight"].map(lambda w: "None" if w is None else w))
for column in ["PCA", "max_features", "min_samples_leaf", "class_weight"]:
    counts = display_selections[column].astype(str).value_counts()
    print(f"  {column}: " + ", ".join(f"{value} in {n}" for value, n in counts.items()))
print(f"  Inner macro F1: mean {selections['Inner macro F1'].mean():.3f} "
      f"(notebook 13 hybrid: {nb13_hybrid['Inner macro F1'].mean():.3f})")

at_new_edge = {"PCA 128": (selections["PCA"] == 128).sum(), "max_features 0.7": (selections["max_features"] == 0.7).sum()}
print("Selections at the new grid edges: " + ", ".join(f"{k} in {v}/25 folds" for k, v in at_new_edge.items()))

Selected settings across 25 outer folds (Protocol Section 5, descriptive):
  PCA: 32 in 14, 16 in 6, 64 in 5
  max_features: 0.7 in 13, 0.5 in 8, 0.3 in 4
  min_samples_leaf: 2 in 15, 1 in 10
  class_weight: balanced in 12
  Inner macro F1: mean 0.767 (notebook 13 hybrid: 0.760)
Selections at the new grid edges: PCA 128 in 0/25 folds, max_features 0.7 in 13/25 folds


## Application model rule (Protocol Section 6)

The rule is applied mechanically.

In [10]:
primary = results[(results["ID"] == "F1") & (results["Metric"] == "Macro F1")].iloc[0]
b_recall = results[(results["ID"] == "F1") & (results["Metric"] == "Recall B")]["Difference"].iloc[0]
convincing = bool(primary["Difference"] > 0 and primary["p"] < 0.05 and b_recall >= -0.02)
print(f"{CANDIDATE} − Tuned RF hybrid: macro F1 {primary['Difference']:+.4f} "
      f"[{primary['CI low']:+.4f}, {primary['CI high']:+.4f}], p = {primary['p']:.4f}, "
      f"B recall difference {b_recall:+.4f}")
if convincing:
    print("Convincingly outperforms the deployed model: a replacement is recommended (not deployed by this notebook).")
else:
    print("Does not convincingly outperform the deployed model: the tuned RF hybrid is retained.")

Extended RF hybrid − Tuned RF hybrid: macro F1 +0.0085 [-0.0160, +0.0330], p = 0.4810, B recall difference +0.0040
Does not convincingly outperform the deployed model: the tuned RF hybrid is retained.


In [11]:
fold_scores.to_csv(ms.PROCESSED_DIR / "extended_tuning_fold_scores.csv", index=False)
results.to_csv(ms.PROCESSED_DIR / "extended_tuning_results.csv", index=False)
selections.to_csv(ms.PROCESSED_DIR / "extended_tuning_selections.csv", index=False)
per_grade.to_csv(ms.PROCESSED_DIR / "extended_tuning_per_grade.csv")
pd.DataFrame({
    "Tissue_ID": np.tile(cohort["Tissue_ID"].to_numpy()[eval_rows], ms.N_REPEATS),
    "Repeat": np.repeat(np.arange(ms.N_REPEATS), len(eval_rows)),
    "Actual": y_eval,
    **pooled,
}).to_csv(ms.PROCESSED_DIR / "extended_tuning_oof_predictions.csv", index=False)
print("Saved fold scores, comparisons, selections, per-grade metrics and out-of-fold predictions to", ms.PROCESSED_DIR)

Saved fold scores, comparisons, selections, per-grade metrics and out-of-fold predictions to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed


## Post-run correction: selected `class_weight`

The "Selected settings" output above is **incomplete for `class_weight`**. It shows only "balanced in 12" because missing values (`class_weight=None`) were dropped by `value_counts()`. This is the same reporting issue as notebook 13.

Recounted from the saved `extended_tuning_selections.csv`, which the notebook wrote from the same in-memory selections: **None in 13, balanced in 12**. No model was refitted. No other result depends on that output.

The other setting counts are complete:

| | Values chosen across 25 folds |
|---|---|
| PCA | 32 in 14, 16 in 6, 64 in 5, 128 in 0 |
| `max_features` | 0.7 in 13, 0.5 in 8, 0.3 in 4 |
| `min_samples_leaf` | 2 in 15, 1 in 10 |

## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| 2026-10-22 | Added the "Post-run correction: selected `class_weight`" cell, recounted from `extended_tuning_selections.csv`. The original output was left in place. No code was re-executed. | Reporting bug: `value_counts()` dropped missing values (`class_weight=None`). No result, comparison or selection depends on this output. |
